In [2]:
import os
import json
import time
import random
import logging
from datasets import load_dataset, Dataset, DatasetDict
from translatepy import Translator
from translatepy.translators import YandexTranslate
from tqdm.auto import tqdm

# Создаём переводчик
yandex = YandexTranslate()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Тест переводчика
test_text = "Hello, world!"
result = yandex.translate(test_text, "ru")
print(f"Original: {test_text}")
print(f"Translated: {result}")

In [4]:
# ---------- Настройки ----------
yandex = YandexTranslate()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

SOURCE_REPO_ID = "DeepPavlov/dialogsum"
LOCAL_SAVE_PATH = "./dialogsum_ru"
CACHE_FILE = "translation_cache_dialogsum.jsonl"
SPLITS = ['train', 'validation', 'test']

In [ ]:
# ---------- Кэш ----------
def load_cache():
    cache = {}
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    cache[obj["text"]] = obj["translation"]
                except:
                    continue
    return cache

def append_cache(text, translation):
    with open(CACHE_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps({"text": text, "translation": translation}, ensure_ascii=False) + "\n")

translation_cache = load_cache()
print(f"Записей в кэше: {len(translation_cache)}")

In [12]:
# -------------------- Улучшенный переводчик --------------------
def is_numeric_string(s: str) -> bool:
    return not bool(re.search(r'[A-Za-zА-Яа-яёЁ]', s))

def translate_text_robust(text, retries=3, delay=3):
    if not isinstance(text, str) or text.strip() == "":
        return "", True
    if is_numeric_string(text):
        return text, True
    if text in translation_cache:
        return translation_cache[text], True

    # Очень длинный текст (>20 000 символов) → разбиваем на абзацы
    if len(text) > 20000:
        logging.info(f"Extremely long text ({len(text)} chars), splitting into paragraphs...")
        paragraphs = re.split(r'\n\s*\n', text)
        if len(paragraphs) > 1:
            translated_paragraphs = []
            all_ok = True
            for para in paragraphs:
                if not para.strip():
                    translated_paragraphs.append(para)
                    continue
                part, ok = translate_text_robust(para, retries=retries, delay=delay)
                if not ok:
                    all_ok = False
                    break
                translated_paragraphs.append(part)
            if all_ok:
                full_trans = '\n\n'.join(translated_paragraphs)
                translation_cache[text] = full_trans
                append_cache(text, full_trans)
                return full_trans, True

    # Длинный текст (>8 000) → разбиваем на предложения
    if len(text) > 8000:
        logging.info(f"Long text ({len(text)} chars), splitting into sentences...")
        sentences = re.split(r'(?<=[.!?])\s+', text)
        if len(sentences) > 1:
            translated_parts = []
            all_ok = True
            for sent in sentences:
                part, ok = translate_text_robust(sent, retries=retries, delay=delay)
                if not ok:
                    all_ok = False
                    break
                translated_parts.append(part)
            if all_ok:
                full_trans = ' '.join(translated_parts)
                translation_cache[text] = full_trans
                append_cache(text, full_trans)
                return full_trans, True

    last_exception = None
    for attempt in range(retries):
        try:
            time.sleep(0.5 if attempt == 0 else delay)
            result = yandex.translate(text, "ru")
            translated = str(result.result) if hasattr(result, 'result') else str(result)
            translation_cache[text] = translated
            append_cache(text, translated)
            return translated, True
        except Exception as e:
            last_exception = e
            error_str = str(e).lower()
            if '413' in error_str or 'too long' in error_str:
                logging.info(f"413 error, splitting: '{text[:30]}...'")
                sentences = re.split(r'(?<=[.!?])\s+', text)
                if len(sentences) > 1:
                    translated_parts = []
                    for sent in sentences:
                        part, ok = translate_text_robust(sent, retries=1, delay=delay)
                        if not ok:
                            break
                        translated_parts.append(part)
                    if len(translated_parts) == len(sentences):
                        full_trans = ' '.join(translated_parts)
                        translation_cache[text] = full_trans
                        append_cache(text, full_trans)
                        return full_trans, True
            if any(code in error_str for code in ['502', '503', '504']):
                wait = delay * (attempt + 1)
                logging.warning(f"Server error '{text[:30]}...' attempt {attempt+1}/{retries}. Wait {wait}s")
                time.sleep(wait)
            elif '429' in error_str:
                wait = delay * 4 + 10
                logging.warning(f"Rate limit. Wait {wait}s")
                time.sleep(wait)
            else:
                time.sleep(delay)

    # Последняя попытка для длинного текста после всех ошибок
    if len(text) > 2000:
        logging.info(f"Final attempt for long text: splitting into sentences")
        sentences = re.split(r'(?<=[.!?])\s+', text)
        if len(sentences) > 1:
            translated_parts = []
            for sent in sentences:
                part, ok = translate_text_robust(sent, retries=1, delay=delay)
                if not ok:
                    break
                translated_parts.append(part)
            if len(translated_parts) == len(sentences):
                full_trans = ' '.join(translated_parts)
                translation_cache[text] = full_trans
                append_cache(text, full_trans)
                return full_trans, True

    logging.error(f"Failed to translate: '{text[:50]}...' Error: {last_exception}")
    return "", False

In [14]:
# -------------------- Перевод диалога с сохранением маркеров --------------------
def translate_dialogue_text(dialogue_str: str):
    """
    Переводит строку диалога, сохраняя маркеры #PersonX# без изменений.
    Возвращает (переведённая_строка, успех).
    """
    if not dialogue_str or not dialogue_str.strip():
        return "", True

    # Разбиваем по маркерам, сохраняя их
    parts = re.split(r'(#Person\w+#:)', dialogue_str)
    translated_parts = []
    all_success = True
    for i, part in enumerate(parts):
        if i % 2 == 0:  # текст между маркерами
            if part.strip():
                trans, ok = translate_text_robust(part)
                if not ok:
                    all_success = False
                translated_parts.append(trans)
            else:
                translated_parts.append(part)
        else:  # маркер
            translated_parts.append(part)

    return ''.join(translated_parts), all_success

# -------------------- Перевод списка сообщений диалога --------------------
def translate_dialog(dialog_list):
    """
    Переводит список сообщений и возвращает список словарей
    только с русским содержанием (name, role, content_ru).
    """
    if not dialog_list:
        return [], True

    translated = []
    all_success = True
    for msg in dialog_list:
        content = msg.get("content", "")
        content_ru, ok = translate_text_robust(content)
        all_success = all_success and ok
        translated.append({
            "name": msg.get("name", ""),
            "role": msg.get("role", ""),
            "content_ru": content_ru
        })
    return translated, all_success

# -------------------- Перевод одного примера --------------------
def translate_example(example):
    success = True

    # dialogue – строка с #Person#, переводим без дублирования оригинала
    dialogue_ru, ok = translate_dialogue_text(example['dialogue'])
    success = success and ok

    summary_ru, ok = translate_text_robust(example['summary'])
    success = success and ok

    topic_ru, ok = translate_text_robust(example['topic'])
    success = success and ok

    # dialog – список, оставляем только русские реплики
    dialog_ru_list, ok = translate_dialog(example['dialog'])
    success = success and ok

    return {
        'id': example['id'],
        'dialogue': example['dialogue'],            # оригинал остаётся
        'dialogue_ru': dialogue_ru,                # только перевод с маркерами
        'summary': example['summary'],
        'summary_ru': summary_ru,
        'topic': example['topic'],
        'topic_ru': topic_ru,
        'dialog': example['dialog'],
        'dialog_ru': dialog_ru_list,               # список {name, role, content_ru}
        '_success': success
    }

# -------------------- Обработка сплита с прогресс-файлом --------------------
def process_split(split_name, source_split, progress_file):
    translated_records = []
    failed_indices = set()

    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    if rec.get('_failed', False):
                        failed_indices.add(rec['_index'])
                    else:
                        translated_records.append(rec)
                except:
                    continue
        logging.info(f"[{split_name}] Resuming. Found {len(translated_records)} successful, {len(failed_indices)} failed.")

    start_index = len(translated_records) + len(failed_indices)
    total = len(source_split)

    if start_index < total:
        logging.info(f"[{split_name}] Starting translation from index {start_index}...")
        with open(progress_file, "a", encoding="utf-8") as f:
            pbar = tqdm(
                enumerate(source_split.select(range(start_index, total))),
                desc=f"Translating {split_name}",
                total=total - start_index
            )
            for idx, example in pbar:
                global_idx = start_index + idx
                if global_idx in {r.get('_index', -1) for r in translated_records}:
                    continue

                translated = translate_example(example)
                record_out = {
                    '_index': global_idx,
                    '_failed': not translated['_success'],
                    **translated
                }
                del record_out['_success']

                f.write(json.dumps(record_out, ensure_ascii=False) + "\n")
                f.flush()
                time.sleep(0.5)

                if not record_out['_failed']:
                    translated_records.append(record_out)
                else:
                    failed_indices.add(global_idx)

    all_successful = []
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    if not rec.get('_failed', False):
                        rec.pop('_index', None)
                        rec.pop('_failed', None)
                        all_successful.append(rec)
                except:
                    continue

    if not all_successful:
        logging.error(f"[{split_name}] No records successfully translated.")
        return None

    ok_cnt = len(all_successful)
    fail_cnt = total - ok_cnt
    logging.info(f"[{split_name}] Translation completed: {ok_cnt}/{total} successful ({fail_cnt} failed)")
    if fail_cnt > 0:
        logging.info(f"[{split_name}] To retry failed, delete/modify {progress_file} and run again.")

    return Dataset.from_list(all_successful)

In [ ]:
# -------------------- Запуск --------------------
logging.info("Loading source dataset...")
source = load_dataset(SOURCE_REPO_ID)

translated_splits = {}
for split in SPLITS:
    if split not in source:
        logging.warning(f"Split '{split}' not found, skipping...")
        continue
    progress_path = f"translated_dialogsum_{split}.jsonl"
    ds_translated = process_split(split, source[split], progress_path)
    if ds_translated is None:
        logging.error(f"Failed to process split {split}")
        break
    translated_splits[split] = ds_translated

if len(translated_splits) == len(SPLITS):
    final_dataset = DatasetDict(translated_splits)
    final_dataset.save_to_disk(LOCAL_SAVE_PATH)
    print(f"\nDataset saved to {LOCAL_SAVE_PATH}")

    # Сравнение с оригиналом
    print("\n" + "="*60)
    print("СРАВНЕНИЕ С ОРИГИНАЛОМ")
    print("="*60)
    for split in SPLITS:
        orig_len = len(source[split])
        our_len = len(final_dataset[split])
        diff = orig_len - our_len
        if diff == 0:
            print(f"{split}: {our_len} (совпадает)")
        else:
            print(f"{split}: оригинал={orig_len}, наш={our_len} (разница {diff})")
else:
    logging.error("Not all splits were successfully translated.")

In [ ]:
import json
from datasets import load_dataset

split = 'test'
progress_file = f"translated_dialogsum_{split}.jsonl"
source = load_dataset("DeepPavlov/dialogsum", split=split)

# Находим индекс проблемной записи
with open(progress_file, 'r', encoding='utf-8') as f:
    for line in f:
        rec = json.loads(line)
        if rec.get('_failed', False):
            failed_idx = rec['_index']
            print(f"Найдена упавшая запись с индексом {failed_idx}")
            break

# Пробуем перевести заново
example = source[int(failed_idx)]
translated = translate_example(example)

# Если снова неудача — просто копируем оригинал в русские поля
if not translated['_success']:
    translated['dialogue_ru'] = example['dialogue']
    translated['summary_ru'] = example['summary']
    translated['topic_ru'] = example['topic']
    translated['dialog_ru'] = [
        {"name": m['name'], "role": m['role'], "content_ru": m['content']}
        for m in example['dialog']
    ]
    translated['_success'] = True   # принудительно помечаем как успех

# Обновляем прогресс-файл
lines = []
with open(progress_file, 'r', encoding='utf-8') as f:
    for line in f:
        rec = json.loads(line)
        if rec.get('_index') == failed_idx:
            new_rec = {'_index': failed_idx, '_failed': False, **{k:v for k,v in translated.items() if k != '_success'}}
            lines.append(json.dumps(new_rec, ensure_ascii=False))
        else:
            lines.append(line.strip())

with open(progress_file, 'w', encoding='utf-8') as f:
    for line in lines:
        f.write(line + '\n')

print("Запись исправлена. Можно удалить служебные столбцы и загружать датасет.")

In [ ]:
from datasets import Dataset, DatasetDict

# Собираем все сплиты без _index и _failed
clean = {}
for split in ['train', 'validation', 'test']:
    with open(f"translated_dialogsum_{split}.jsonl", 'r', encoding='utf-8') as f:
        recs = [json.loads(line) for line in f]
    for r in recs:
        r.pop('_index', None)
        r.pop('_failed', None)
    clean[split] = Dataset.from_list(recs)

final_dataset = DatasetDict(clean)
final_dataset.save_to_disk("./dialogsum_ru")
print("Чистый датасет сохранён в ./dialogsum_ru")

In [ ]:
from datasets import DatasetDict
from huggingface_hub import login

login(token="YOUR_HF_TOKEN")
dataset = DatasetDict.load_from_disk("./dialogsum_ru")
REPO_ID = "DeepPavlov/dialogsum_ru"

for split_name, ds in dataset.items():
    ds.push_to_hub(
        REPO_ID,
        config_name=split_name,
        private=False,
        commit_message="DialogSum Russian translation"
    )
    print(f"{split_name} uploaded")

In [ ]:
import json, re
from datasets import Dataset, DatasetDict
from huggingface_hub import login

login(token="YOUR_HF_TOKEN")
REPO_ID = "DeepPavlov/dialogsum_ru"
SPLITS = ['train', 'validation', 'test']

def fix_markers(text):
    """Заменяет русские Персонаж/Человек обратно на Person"""
    if not isinstance(text, str):
        return text
    # Возможные варианты перевода маркера
    text = re.sub(r'#Персонаж(\d+)#', r'#Person\1#', text)
    text = re.sub(r'#Человек(\d+)#', r'#Person\1#', text)
    text = re.sub(r'#Persona(\d+)#', r'#Person\1#', text)
    return text

for split in SPLITS:
    progress_file = f"translated_dialogsum_{split}.jsonl"
    fixed_records = []
    
    with open(progress_file, 'r', encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            
            # Исправляем dialogue_ru
            if 'dialogue_ru' in rec:
                rec['dialogue_ru'] = fix_markers(rec['dialogue_ru'])
            
            # Исправляем summary_ru
            if 'summary_ru' in rec:
                rec['summary_ru'] = fix_markers(rec['summary_ru'])
            
            # Исправляем topic_ru (на всякий случай)
            if 'topic_ru' in rec:
                rec['topic_ru'] = fix_markers(rec['topic_ru'])
            
            # Исправляем dialog_ru (список сообщений)
            if 'dialog_ru' in rec and isinstance(rec['dialog_ru'], list):
                for msg in rec['dialog_ru']:
                    if 'content_ru' in msg:
                        msg['content_ru'] = fix_markers(msg['content_ru'])
            
            fixed_records.append(rec)
    
    # Сохраняем исправленный прогресс-файл
    with open(progress_file, 'w', encoding='utf-8') as f:
        for rec in fixed_records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    
    print(f"✅ {split}: исправлены маркеры")

# Пересобираем датасет
clean = {}
for split in SPLITS:
    with open(f"translated_dialogsum_{split}.jsonl", 'r', encoding='utf-8') as f:
        recs = [json.loads(line) for line in f]
    for r in recs:
        r.pop('_index', None)
        r.pop('_failed', None)
    clean[split] = Dataset.from_list(recs)
    print(f"{split}: {len(clean[split])} записей")

final_dataset = DatasetDict(clean)
final_dataset.save_to_disk("./dialogsum_ru")
print("✅ Датасет сохранён")

✅ train: исправлены маркеры
✅ validation: исправлены маркеры
✅ test: исправлены маркеры
train: 12460 записей
validation: 500 записей
test: 1500 записей


Saving the dataset (1/1 shards): 100%|██████████| 1500/1500 [00:00<00:00, 186242.44 examples/s]

✅ Датасет сохранён


In [25]:
from datasets import DatasetDict
import json

# Загружаем локальный датасет
dataset = DatasetDict.load_from_disk("./dialogsum_ru")

# Смотрим примеры из каждого сплита
for split in ['train', 'validation', 'test']:
    print(f"\n{'='*60}")
    print(f"СПЛИТ: {split}")
    print('='*60)
    
    # Берём случайный пример (или конкретный индекс)
    sample = dataset[split][2]  # третий пример
    
    print(f"id: {sample['id']}")
    print(f"\ndialogue_ru (первые 300 символов):")
    print(sample['dialogue_ru'][:300])
    
    print(f"\nsummary_ru:")
    print(sample['summary_ru'])
    
    print(f"\ntopic_ru:")
    print(sample['topic_ru'])
    
    print(f"\ndialog_ru (первые 2 сообщения):")
    for msg in sample['dialog_ru'][:2]:
        print(f"  {msg['name']} ({msg['role']}): {msg['content_ru'][:100]}...")
    
    # Проверяем, остались ли русские маркеры
    text_to_check = sample['dialogue_ru'] + ' ' + sample['summary_ru']
    if 'Персонаж' in text_to_check or 'Человек' in text_to_check or 'Persona' in text_to_check:
        print("\n❌ НАЙДЕНЫ РУССКИЕ МАРКЕРЫ!")
    else:
        print("\n✅ Маркеры в порядке (только #Person#)")
    
    print()


СПЛИТ: train
id: train_2

dialogue_ru (первые 300 символов):
#Person1#: Извините, вы не видели связку ключей?
#Person2#: Что это за ключи?
#Person1#: Пять ключей и маленькое украшение для ног.
#Person2#: Какая жалость! Я их не видел.
#Person1#: Не могли бы вы помочь мне найти его? Я здесь впервые.
#Person2#: Конечно. С удовольствием. Я бы хотел помочь вам най

summary_ru:
#Person1# ищет связку ключей и просит #Человека 2# помочь ему найти их.

topic_ru:
найти ключи

dialog_ru (первые 2 сообщения):
  Person1 (user): Извините, вы не видели связку ключей?...
  Person2 (assistant): Что это за ключи?...

❌ НАЙДЕНЫ РУССКИЕ МАРКЕРЫ!


СПЛИТ: validation
id: dev_2

dialogue_ru (первые 300 символов):
#Person1#: Мне нужно перестать есть такую нездоровую пищу.
#Person2#: Я понимаю, что вы имеете в виду. Я и сам стал лучше питаться.
#Person1#: Какие продукты вы едите сейчас?
#Person2#: Я предпочитаю фрукты, овощи и курицу.
#Person1#: Это все, что ты ешь?
#Person2#: Это в основном то, что я ем.
#Pe

In [26]:
# Загружаем на HF
for split_name, ds in final_dataset.items():
    ds.push_to_hub(
        REPO_ID,
        config_name=split_name,
        private=False,
        commit_message="Fixed #Person# markers in Russian translations"
    )
    print(f"✅ {split_name} uploaded")

print(f"\n🎉 Готово: https://huggingface.co/datasets/{REPO_ID}")

Creating parquet from Arrow format: 100%|██████████| 13/13 [00:00<00:00, 66.81ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
2026-05-07 19:24:14,973 - WARNING - Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Uploading the dataset shards: 100%|██████████| 1/1 [01:51<00:00, 111.46s/it]


✅ train uploaded


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 21.66ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
2026-05-07 19:26:08,502 - WARNING - Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.88s/it]


✅ validation uploaded


Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 84.90ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
2026-05-07 19:26:13,874 - WARNING - Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.81s/it]


✅ test uploaded

🎉 Готово: https://huggingface.co/datasets/DeepPavlov/dialogsum_ru
